In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import accuracy_score

In [ ]:
import pandas as pd
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [ ]:
def fillna(df):
    df = df.copy()  # 명시적 복사
    df['Age'] = df['Age'].fillna(df['Age'].mean())
    df['Cabin'] = df['Cabin'].fillna('N')
    df['Embarked'] = df['Embarked'].fillna('N')
    df['Fare'] = df['Fare'].fillna(0)
    return df

def drop_features(df):
    return df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)

def format_features(df):
    df = df.copy()
    df['Cabin'] = df['Cabin'].str[:1]
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        df[feature] = le.fit_transform(df[feature])
    return df

# 앞에서 설정한 데이터 전처리 함수 호출
def transform_features(df):
    df = fillna(df)
    df = drop_features(df)
    df = format_features(df)
    return df

In [ ]:
def get_clf_eval(y_test, pred):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)

    print(confusion)
    print('*'*20)
    print(accuracy, precision, recall)

In [ ]:
titanic_df = transform_features(df)

# 피처와 타겟 분리

In [ ]:
y_titanic_df = titanic_df['Survived']
X_titanic_df = titanic_df.drop('Survived', axis=1)

# 훈련, 테스트 데이터 분리

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_titanic_df,
                                                    y_titanic_df,
                                                    test_size=0.2,
                                                    random_state=0 )

# 성능비교 - 로지스틱회귀

In [ ]:
#로지스틱회귀 분류모델 생성
from sklearn.linear_model import LogisticRegression

lr_clf = LogisticRegression(max_iter=2000)
lr_clf.fit(X_train, y_train)
pred = lr_clf.predict(X_test)

#정확도, 정밀도, 재현율
get_clf_eval(y_test, pred)

[[92 18]
 [16 53]]
********************
0.8100558659217877 0.7464788732394366 0.7681159420289855


# 단순가설의 분류기를 이용한 성능 기준 확인

남성 > 사망, 여성 > 생존


In [ ]:
from sklearn.base import BaseEstimator
import numpy as np
class MyDummyClassifier(BaseEstimator):
  def fit(self, X, y):
    pass

  def predict(self, X):
    pred = np.zeros((X.shape[0],1))
    for i in range(X.shape[0]):
      if X['Sex'].iloc[i] == 1:
        pred[i]=0
      else :
        pred[i]=1
    return pred

In [ ]:
myclf = MyDummyClassifier()
myclf.fit(X_train, y_train)
my_pred = myclf.predict(X_test)
accuracy_score(y_test, my_pred)

0.7877094972067039

# 랜덤포레스트, KNN 의 정밀도, 재현율 비교하기

In [ ]:
# 랜덤포레스트, KNN 의 정밀도, 재현율 비교하기
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# ── 1. 랜덤포레스트 ──────────────────────────────────────
rf_clf = RandomForestClassifier(n_estimators=100, random_state=0)
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)

print("=" * 40)
print("[ 랜덤포레스트 ]")
get_clf_eval(y_test, rf_pred)

# ── 2. KNN ───────────────────────────────────────────────
knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(X_train, y_train)
knn_pred = knn_clf.predict(X_test)

print("=" * 40)
print("[ KNN (k=5) ]")
get_clf_eval(y_test, knn_pred)

# ── 3. 세 모델 성능 비교 요약 ────────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score

models = {
    "로지스틱 회귀" : pred,      # 앞서 lr_clf로 구한 pred
    "랜덤포레스트"  : rf_pred,
    "KNN (k=5)"    : knn_pred,
}

print("\n" + "=" * 40)
print(f"{'모델':<15} {'정확도':>8} {'정밀도':>8} {'재현율':>8}")
print("-" * 40)
for name, p in models.items():
    acc  = accuracy_score (y_test, p)
    prec = precision_score(y_test, p)
    rec  = recall_score   (y_test, p)
    print(f"{name:<15} {acc:>8.4f} {prec:>8.4f} {rec:>8.4f}")

[ 랜덤포레스트 ]
[[99 11]
 [20 49]]
********************
0.8268156424581006 0.8166666666666667 0.7101449275362319
[ KNN (k=5) ]
[[94 16]
 [31 38]]
********************
0.7374301675977654 0.7037037037037037 0.5507246376811594

모델                   정확도      정밀도      재현율
----------------------------------------
로지스틱 회귀           0.8101   0.7465   0.7681
랜덤포레스트            0.8268   0.8167   0.7101
KNN (k=5)         0.7374   0.7037   0.5507


# 예상 결과 해석 방향

정밀도(Precision) → 생존 예측 중 실제 생존자 비율 → 높을수록 "헛된 희망"을 줄임
재현율(Recall) → 실제 생존자 중 잡아낸 비율 → 높을수록 생존자를 놓치지 않음
일반적으로 랜덤포레스트가 로지스틱 회귀 · KNN 대비 정확도가 가장 높게 나옵니다

# 분류모델의 임계치 확인

In [ ]:
pred_proba = lr_clf.predict_proba(X_test)
pos_proba = pred_proba[:,1] #양성일 확률값

threshold = 0.4
custom_proba = (pos_proba >= threshold).astype(int)
confusion_matrix(y_test, custom_proba)

array([[86, 24],
       [13, 56]])

In [ ]:
get_clf_eval(y_test, custom_proba)

[[86 24]
 [13 56]]
********************
0.7932960893854749 0.7 0.8115942028985508
